# Windowing-Based Machine Learning
## Wind Power Forecasting | Seminar WIBA

**Idee:** Zeitreihe → Cross-Sectional Datensatz mit zeitverzögerten Inputs (Lags).
Jede Zeile enthält vergangene Werte als Features und den zukünftigen Wert als Target.

$$X_t = [\omega_{t-1}, \omega_{t-2}, \ldots, \omega_{t-W}], \quad y_t = \omega_{t+H}$$

| Parameter | Bedeutung |
|-----------|----------|
| **W** (Window Size) | Wie viele vergangene Stunden als Features |
| **H** (Horizon) | Wie weit in die Zukunft vorhergesagt wird |
| **S** (Skip) | Schrittweite zwischen Fenstern |

**Modelle in diesem Notebook:**
1. Linear Regression
2. Elastic Net
3. Random Forest
4. Gradient Boosting (GBRT)
5. Neural Network (MLP)
6. LSTM

---

## 1. Imports und Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression, ElasticNetCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(42)
torch.manual_seed(42)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size']  = 11
sns.set_style('whitegrid')
print('Imports erfolgreich.')

## 2. Daten laden

In [ ]:
DATA_PATH = '/Users/noahweis/wind_bidding_project/data/processed/final_dataset.csv'

df = pd.read_csv(DATA_PATH, parse_dates=['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

TARGET = 'power'

print(f'Datensatz: {df.shape[0]} Zeilen, {df.shape[1]} Spalten')
print(f'Zeitraum:  {df.timestamp.min()} bis {df.timestamp.max()}')
print(f'Spalten:   {df.columns.tolist()}')
df.head()

## 3. Windowing – Cross-Sectional Datensatz erstellen

### 3.1 Konzept

```
Zeitreihe:  [ω₁, ω₂, ω₃, ω₄, ω₅, ω₆, ω₇, ...]

Window W=3, Horizon H=1, Skip S=1:

  Row 1:  X=[ω₁, ω₂, ω₃]  →  y=ω₄
  Row 2:  X=[ω₂, ω₃, ω₄]  →  y=ω₅
  Row 3:  X=[ω₃, ω₄, ω₅]  →  y=ω₆
  ...
```

Das Windowing wandelt das Zeitreihenproblem in ein Standard-Supervised-Learning-Problem um.
Danach kann **jedes beliebige ML-Modell** angewendet werden – das ist der zentrale Vorteil.

### 3.2 Willkürlichkeit der Parameter
- **W zu klein**: zu wenig Kontext, Modell sieht keine saisonalen Muster
- **W zu groß**: viele korrelierte Features, langsamer, Overfitting-Risiko
- **H**: bestimmt den Forecasting-Horizont – für Day-Ahead Bidding: H=1 bis H=24
- **S**: S=1 → überlappende Fenster (Standard); S=W → nicht-überlappende Fenster

### 3.3 Windowing-Parameter

In [ ]:
# ── Parameter anpassen nach Bedarf ──────────────────────────────────────
W = 24   # Window Size: 24 Stunden Vergangenheit (= 1 Tag)
H = 1    # Forecast Horizon: 1 Stunde voraus
S = 1    # Skip: jede Stunde ein Sample

print(f'Window Size W = {W}  → Features: lag_1 bis lag_{W}')
print(f'Horizon     H = {H}  → Target: ω_{{t+{H}}}')
print(f'Skip        S = {S}  → Schrittweite zwischen Samples')
print(f'Erwartete Samples: ~{(len(df) - W - H) // S}')

### 3.4 Windowing-Funktion

In [ ]:
def create_windowed_dataset(series: np.ndarray,
                             W: int, H: int, S: int = 1,
                             extra_features: np.ndarray = None):
    """
    Erstellt Cross-Sectional Datensatz aus Zeitreihe via Sliding Window.

    Args:
        series:         1D Zeitreihe der Zielvariable (N,)
        W:              Window Size – Anzahl Lag-Features
        H:              Forecast Horizon
        S:              Skip / Schrittweite
        extra_features: zusätzliche Features pro Zeitpunkt (N, F) – optional

    Returns:
        X: (n_samples, W + F) Feature-Matrix
        y: (n_samples,)       Target-Vektor
        indices: Zeitindex der Targets (für spätere Plots)
    """
    X_list, y_list, idx_list = [], [], []

    for i in range(0, len(series) - W - H + 1, S):
        # Lag-Features: W vergangene Werte
        lag_features = series[i : i + W]

        # Optionale zusätzliche Features (z.B. Windgeschwindigkeit, Zeitfeatures)
        if extra_features is not None:
            extra = extra_features[i + W - 1]  # aktueller Zeitpunkt
            row = np.concatenate([lag_features, extra])
        else:
            row = lag_features

        # Target: H Schritte voraus
        target = series[i + W + H - 1]

        X_list.append(row)
        y_list.append(target)
        idx_list.append(i + W + H - 1)

    return np.array(X_list), np.array(y_list), np.array(idx_list)

print('Windowing-Funktion definiert.')

### 3.5 Datensatz erstellen

In [ ]:
# Zielvariable
power_series = df[TARGET].values

# Optionale Zusatz-Features (alle numerischen Spalten außer Target)
numeric_cols   = df.select_dtypes(include=[np.number]).columns.tolist()
extra_cols     = [c for c in numeric_cols if c != TARGET]
extra_features = df[extra_cols].values if extra_cols else None

print(f'Zusatz-Features: {extra_cols}')

# Windowed Dataset erstellen
X_all, y_all, indices = create_windowed_dataset(
    power_series, W=W, H=H, S=S, extra_features=extra_features
)

print(f'\nX Shape: {X_all.shape}  (Samples x Features)')
print(f'y Shape: {y_all.shape}')
print(f'Feature-Spalten: {W} Lags + {len(extra_cols)} Zusatz-Features = {X_all.shape[1]} total')

# Feature-Namen für spätere Analyse
lag_names   = [f'lag_{i}' for i in range(1, W + 1)]
feature_names = lag_names + extra_cols
print(f'Erste Features: {feature_names[:5]} ... {feature_names[-3:]}')

### 3.6 Visualisierung der Fenster-Struktur

Kurze visuelle Überprüfung: Sieht der Datensatz sinnvoll aus?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Beispiel: erste 3 Fenster visualisieren
for i in range(3):
    axes[0].plot(range(i, i + W), X_all[i, :W],
                 label=f'Fenster {i+1}', alpha=0.8, linewidth=1.5)
    axes[0].scatter([i + W], [y_all[i]], marker='*', s=100)
axes[0].set_title(f'Windowing-Beispiel: W={W}, H={H}')
axes[0].set_xlabel('Zeitindex')
axes[0].set_ylabel('Leistung [kW]')
axes[0].legend(fontsize=8)

# Verteilung der Target-Werte
axes[1].hist(y_all, bins=50, color='#4C72B0', edgecolor='white', alpha=0.8)
axes[1].set_title('Verteilung der Target-Werte')
axes[1].set_xlabel('Leistung [kW]')
axes[1].set_ylabel('Häufigkeit')

plt.tight_layout()
plt.savefig('../results/figures/wml_windowing_example.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Train/Test Split (80/20)

**Wichtig:** Zeitreihen-konformer Split – kein Shuffle!
Die ersten 80% der Samples werden zum Training verwendet, die letzten 20% zum Testen.

In [ ]:
TEST_SIZE = 0.2
split_idx = int(len(X_all) * (1 - TEST_SIZE))

X_train, X_test = X_all[:split_idx], X_all[split_idx:]
y_train, y_test = y_all[:split_idx], y_all[split_idx:]
idx_train, idx_test = indices[:split_idx], indices[split_idx:]

# Skalierung (Pflicht für LR, EN, NN, LSTM)
scaler  = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_te_sc = scaler.transform(X_test)

print(f'Train: {len(X_train)} Samples ({100*(1-TEST_SIZE):.0f}%)')
print(f'Test:  {len(X_test)}  Samples ({100*TEST_SIZE:.0f}%)')
print(f'Features pro Sample: {X_train.shape[1]}')

## 5. Hilfsfunktionen: Metriken

In [ ]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def mae(y_true, y_pred):
    return mean_absolute_error(y_true, y_pred)

def mape(y_true, y_pred, eps=1e-6):
    mask = np.abs(y_true) > eps
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def newsvendor_loss(y_true, y_bid, c_over=10.0, c_under=15.0):
    over  = np.maximum(y_bid - y_true, 0)
    under = np.maximum(y_true - y_bid, 0)
    return np.mean(c_over * over + c_under * under)

def evaluate(y_true, y_pred):
    return {
        'RMSE':            round(rmse(y_true, y_pred), 3),
        'MAE':             round(mae(y_true, y_pred), 3),
        'MAPE (%)':        round(mape(y_true, y_pred), 3),
        'Newsvendor Loss': round(newsvendor_loss(y_true, y_pred), 3),
    }

results     = {}
predictions = {}
print('Metriken definiert.')

## 6. Modell 1 – Linear Regression

$$\hat{\omega} = \beta_0 + \sum_{k=1}^{W} \beta_k \cdot \omega_{t-k} + \text{Zusatz-Features}$$

Interpretierbarste Variante. Lag-Koeffizienten zeigen, welche Vergangenheitswerte am wichtigsten sind.

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_tr_sc, y_train)
y_pred_lr = np.clip(lr_model.predict(X_te_sc), 0, None)

results['LinearRegression']     = evaluate(y_test, y_pred_lr)
predictions['LinearRegression'] = y_pred_lr

print('Linear Regression:')
for k, v in results['LinearRegression'].items(): print(f'  {k}: {v}')

In [ ]:
# Koeffizienten der Lag-Features visualisieren
coef_series = pd.Series(lr_model.coef_, index=feature_names)
lag_coefs   = coef_series[lag_names]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(range(1, W + 1), lag_coefs.values, color='#4C72B0', alpha=0.8)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title('LR Lag-Koeffizienten – Einfluss vergangener Stunden')
axes[0].set_xlabel('Lag (Stunden zurück)')
axes[0].set_ylabel('Koeffizient β')

if extra_cols:
    extra_coefs = coef_series[extra_cols].sort_values(key=abs, ascending=True)
    extra_coefs.plot(kind='barh', ax=axes[1], color='#55A868')
    axes[1].axvline(0, color='black', linewidth=0.8)
    axes[1].set_title('LR Koeffizienten – Zusatz-Features')

plt.tight_layout()
plt.savefig('../results/figures/wml_lr_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Modell 2 – Elastic Net

$$\hat{\beta}^{EN} = \arg\min_{\beta} \left\{ \|y - X\beta\|^2 + \lambda_1 \sum_j |\beta_j| + \lambda_2 \sum_j \beta_j^2 \right\}$$

Besonders geeignet wenn Lag-Features stark korreliert sind (benachbarte Lags).
Setzt irrelevante Lags exakt auf 0 → automatische Lag-Selektion.

In [ ]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=False)

en_model = ElasticNetCV(
    l1_ratio  = [0.1, 0.3, 0.5, 0.7, 0.9, 1.0],
    alphas    = np.logspace(-4, 1, 30),
    cv        = kf,
    max_iter  = 5000,
    n_jobs    = -1
)
en_model.fit(X_tr_sc, y_train)
y_pred_en = np.clip(en_model.predict(X_te_sc), 0, None)

results['ElasticNet']     = evaluate(y_test, y_pred_en)
predictions['ElasticNet'] = y_pred_en

print(f'Optimales alpha:    {en_model.alpha_:.5f}')
print(f'Optimales l1_ratio: {en_model.l1_ratio_:.2f}')
n_active = (np.abs(en_model.coef_) > 1e-6).sum()
print(f'Aktive Features:    {n_active}/{X_train.shape[1]}')
print()
for k, v in results['ElasticNet'].items(): print(f'  {k}: {v}')

In [ ]:
# Welche Lags wurden aktiviert?
en_coefs  = pd.Series(en_model.coef_, index=feature_names)
lag_en    = en_coefs[lag_names]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
colors = ['#C44E52' if abs(v) > 1e-6 else '#CCCCCC' for v in lag_en.values]
axes[0].bar(range(1, W + 1), lag_en.values, color=colors, alpha=0.9)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title(f'Elastic Net – Lag-Koeffizienten\n(grau = auf 0 gesetzt, l1_ratio={en_model.l1_ratio_:.1f})')
axes[0].set_xlabel('Lag (Stunden zurück)')
axes[0].set_ylabel('Koeffizient β')

axes[1].plot(range(1, W+1), lag_coefs.values, label='Linear Regression', linewidth=1.5)
axes[1].plot(range(1, W+1), lag_en.values,    label='Elastic Net',       linewidth=1.5, linestyle='--')
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].set_title('LR vs. EN – Lag-Koeffizienten Vergleich')
axes[1].set_xlabel('Lag')
axes[1].legend()

plt.tight_layout()
plt.savefig('../results/figures/wml_en_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Modell 3 – Random Forest

$$\hat{\omega} = \frac{1}{B} \sum_{b=1}^{B} T_b(X_t)$$

Kann nichtlineare Zusammenhänge zwischen Lags und Produktion abbilden.
Feature Importance zeigt welche Lags am informativsten sind.

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators = 200,
    max_depth    = None,
    random_state = 42,
    n_jobs       = -1
)
rf_model.fit(X_train, y_train)
y_pred_rf = np.clip(rf_model.predict(X_test), 0, None)

results['RandomForest']     = evaluate(y_test, y_pred_rf)
predictions['RandomForest'] = y_pred_rf

print('Random Forest:')
for k, v in results['RandomForest'].items(): print(f'  {k}: {v}')

In [ ]:
importance    = pd.Series(rf_model.feature_importances_, index=feature_names)
lag_imp       = importance[lag_names]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(range(1, W + 1), lag_imp.values, color='#55A868', alpha=0.8)
axes[0].set_title('Random Forest – Lag Feature Importance')
axes[0].set_xlabel('Lag (Stunden zurück)')
axes[0].set_ylabel('Importance')

importance.sort_values(ascending=True).tail(10).plot(kind='barh', ax=axes[1], color='#55A868', alpha=0.8)
axes[1].set_title('Top-10 Features (alle)')

plt.tight_layout()
plt.savefig('../results/figures/wml_rf_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Modell 4 – Gradient Boosting (GBRT)

$$\hat{\omega} = \sum_{m=1}^{M} f_m(X_t), \quad f_m \text{ korrigiert Fehler von } f_{m-1}$$

Sequentielle Bäume – jeder Baum lernt die Residuen des vorherigen.
Laut Literatur vergleichbar mit Deep Learning wenn in Windowing-Framework eingebettet.

In [ ]:
gb_model = GradientBoostingRegressor(
    n_estimators  = 200,
    learning_rate = 0.05,
    max_depth     = 4,
    random_state  = 42
)
gb_model.fit(X_train, y_train)
y_pred_gb = np.clip(gb_model.predict(X_test), 0, None)

results['GradientBoosting']     = evaluate(y_test, y_pred_gb)
predictions['GradientBoosting'] = y_pred_gb

print('Gradient Boosting:')
for k, v in results['GradientBoosting'].items(): print(f'  {k}: {v}')

In [ ]:
gb_imp  = pd.Series(gb_model.feature_importances_, index=feature_names)
gb_lag  = gb_imp[lag_names]

fig, ax = plt.subplots(figsize=(12, 4))
x = np.arange(W)
ax.bar(x - 0.2, lag_imp.values, width=0.4, label='Random Forest', color='#55A868', alpha=0.8)
ax.bar(x + 0.2, gb_lag.values, width=0.4, label='Gradient Boosting', color='#C44E52', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels([f'lag_{i+1}' for i in x], rotation=45, fontsize=8)
ax.set_title('Lag Feature Importance: Random Forest vs. Gradient Boosting')
ax.set_ylabel('Importance')
ax.legend()
plt.tight_layout()
plt.savefig('../results/figures/wml_rf_vs_gb_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Modell 5 – Neural Network (MLP)

Vollvernetztes Netz auf dem Windowing-Feature-Vektor.

Architektur: `[W + extra_features] → 128 → 64 → 32 → 1`

Braucht skalierte Features (X_tr_sc).

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(128, 64),        nn.ReLU(),
            nn.Linear(64, 32),         nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

X_t    = torch.FloatTensor(X_tr_sc)
y_t    = torch.FloatTensor(y_train)
loader = DataLoader(TensorDataset(X_t, y_t), batch_size=128, shuffle=True)

mlp       = MLP(X_tr_sc.shape[1])
optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.MSELoss()

losses = []
for epoch in range(100):
    mlp.train()
    ep_loss = 0
    for xb, yb in loader:
        optimizer.zero_grad()
        loss = criterion(mlp(xb), yb)
        loss.backward()
        optimizer.step()
        ep_loss += loss.item()
    losses.append(ep_loss / len(loader))
    if (epoch + 1) % 20 == 0:
        print(f'Epoch {epoch+1:3d}/100 – Loss: {losses[-1]:.4f}')

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(losses, color='#C44E52')
ax.set_title('MLP Trainingsverlust')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE')
plt.tight_layout()
plt.show()

In [ ]:
mlp.eval()
with torch.no_grad():
    y_pred_mlp = mlp(torch.FloatTensor(X_te_sc)).numpy()
y_pred_mlp = np.clip(y_pred_mlp, 0, None)

results['MLP']     = evaluate(y_test, y_pred_mlp)
predictions['MLP'] = y_pred_mlp

print('Neural Network (MLP):')
for k, v in results['MLP'].items(): print(f'  {k}: {v}')

## 11. Modell 6 – LSTM

Das LSTM (Long Short-Term Memory) verarbeitet die Sequenz direkt – kein flacher Feature-Vektor.
Die W Lag-Werte werden als Zeitreihe [W, 1] übergeben, nicht als flacher Vektor.

$$h_t = \text{LSTM}(\omega_{t-W}, \ldots, \omega_{t-1}) \rightarrow \hat{\omega}_{t+H}$$

Vorteil: Lernt zeitliche Abhängigkeiten intern – kein explizites Feature-Engineering nötig.

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size  = input_size,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = 0.1
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze(-1)

X_seq_train = X_train[:, :W].reshape(-1, W, 1).astype(np.float32)
X_seq_test  = X_test[:, :W].reshape(-1, W, 1).astype(np.float32)

seq_scaler = StandardScaler()
X_seq_flat_train = seq_scaler.fit_transform(X_train[:, :W])
X_seq_flat_test  = seq_scaler.transform(X_test[:, :W])
X_seq_train_sc   = X_seq_flat_train.reshape(-1, W, 1).astype(np.float32)
X_seq_test_sc    = X_seq_flat_test.reshape(-1, W, 1).astype(np.float32)

y_scaler    = StandardScaler()
y_train_sc  = y_scaler.fit_transform(y_train.reshape(-1, 1)).flatten().astype(np.float32)

lstm_ds     = TensorDataset(torch.FloatTensor(X_seq_train_sc), torch.FloatTensor(y_train_sc))
lstm_loader = DataLoader(lstm_ds, batch_size=128, shuffle=True)

lstm_model = LSTMModel(input_size=1, hidden_size=64, num_layers=2)
lstm_opt   = torch.optim.Adam(lstm_model.parameters(), lr=1e-3)
lstm_crit  = nn.MSELoss()

lstm_losses = []
for epoch in range(100):
    lstm_model.train()
    ep_loss = 0
    for xb, yb in lstm_loader:
        lstm_opt.zero_grad()
        loss = lstm_crit(lstm_model(xb), yb)
        loss.backward()
        lstm_opt.step()
        ep_loss += loss.item()
    lstm_losses.append(ep_loss / len(lstm_loader))
    if (epoch + 1) % 20 == 0:
        print(f'Epoch {epoch+1:3d}/100 – Loss: {lstm_losses[-1]:.6f}')

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(lstm_losses, color='#8172B2')
ax.set_title('LSTM Trainingsverlust')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE (skaliert)')
plt.tight_layout()
plt.show()

In [ ]:
lstm_model.eval()
with torch.no_grad():
    y_pred_lstm_sc = lstm_model(torch.FloatTensor(X_seq_test_sc)).numpy()

y_pred_lstm = y_scaler.inverse_transform(y_pred_lstm_sc.reshape(-1, 1)).flatten()
y_pred_lstm = np.clip(y_pred_lstm, 0, None)

results['LSTM']     = evaluate(y_test, y_pred_lstm)
predictions['LSTM'] = y_pred_lstm

print('LSTM:')
for k, v in results['LSTM'].items(): print(f'  {k}: {v}')

## 12. Modellvergleich

### 12.1 Metriken-Tabelle

In [ ]:
df_results = pd.DataFrame(results).T.astype(float).round(3)

print('── Windowing ML – Modellvergleich ───────────────────────')
print(df_results.to_string())
print()
print('Bestes RMSE:            ', df_results['RMSE'].idxmin())
print('Bestes Newsvendor Loss: ', df_results['Newsvendor Loss'].idxmin())

df_results.to_csv('../results/tables/windowing_ml_metrics.csv')
print('\nGespeichert: results/tables/windowing_ml_metrics.csv')

### 12.2 Balkendiagramm

In [ ]:
model_colors = ['#4C72B0','#DD8452','#55A868','#C44E52','#8172B2','#937860']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

df_results['RMSE'].sort_values().plot(kind='barh', ax=axes[0], color=model_colors, edgecolor='white')
axes[0].set_title('RMSE (niedriger = besser)')
axes[0].set_xlabel('RMSE [kW]')

df_results['Newsvendor Loss'].sort_values().plot(kind='barh', ax=axes[1], color=model_colors, edgecolor='white')
axes[1].set_title('Newsvendor Loss (niedriger = besser)')
axes[1].set_xlabel('Loss')

plt.suptitle('Windowing ML – Modellvergleich', fontsize=13)
plt.tight_layout()
plt.savefig('../results/figures/wml_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 12.3 Forecast vs. Actual – alle Modelle (2 Wochen)

In [ ]:
N_PLOT = 336
colors_map = {
    'LinearRegression': '#4C72B0',
    'ElasticNet':       '#DD8452',
    'RandomForest':     '#55A868',
    'GradientBoosting': '#C44E52',
    'MLP':              '#8172B2',
    'LSTM':             '#937860',
}

fig, axes = plt.subplots(len(predictions), 1, figsize=(14, 3*len(predictions)), sharex=True)
for ax, (name, y_pred) in zip(axes, predictions.items()):
    ax.plot(y_test[:N_PLOT], label='Actual', color='black', linewidth=1.2, alpha=0.9)
    ax.plot(y_pred[:N_PLOT], label=name, color=colors_map[name], linewidth=1.2, linestyle='--')
    ax.set_ylabel('kW')
    ax.set_title(f"{name}  |  RMSE={results[name]['RMSE']:.1f}  NV-Loss={results[name]['Newsvendor Loss']:.1f}")
    ax.legend(loc='upper right', fontsize=8)
axes[-1].set_xlabel('Stunde (Testperiode)')
plt.suptitle('Windowing ML – Forecast vs. Actual (erste 2 Wochen)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../results/figures/wml_forecast_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()

### 12.4 Window Size Sensitivitätsanalyse

Wie stark beeinflusst W die Performance? – Wichtig weil W willkürlich ist.

In [ ]:
W_values = [6, 12, 24, 48, 72]
sensitivity = []

for w in W_values:
    X_w, y_w, _ = create_windowed_dataset(power_series, W=w, H=H, S=S)
    sp = int(len(X_w) * 0.8)
    rf_tmp = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    rf_tmp.fit(X_w[:sp], y_w[:sp])
    y_tmp = rf_tmp.predict(X_w[sp:])
    sensitivity.append({
        'W': w,
        'RMSE': round(rmse(y_w[sp:], y_tmp), 2),
        'NV-Loss': round(newsvendor_loss(y_w[sp:], y_tmp), 2)
    })
    print(f'W={w:3d}: RMSE={sensitivity[-1]["RMSE"]:.2f}')

df_sens = pd.DataFrame(sensitivity)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(df_sens['W'], df_sens['RMSE'], marker='o', color='#4C72B0', linewidth=2)
axes[0].set_title('Window Size vs. RMSE (Random Forest)')
axes[0].set_xlabel('Window Size W')
axes[0].set_ylabel('RMSE [kW]')

axes[1].plot(df_sens['W'], df_sens['NV-Loss'], marker='o', color='#C44E52', linewidth=2)
axes[1].set_title('Window Size vs. Newsvendor Loss')
axes[1].set_xlabel('Window Size W')
axes[1].set_ylabel('NV-Loss')

plt.suptitle('Sensitivitätsanalyse: Window Size W', fontsize=12)
plt.tight_layout()
plt.savefig('../results/figures/wml_window_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nSensitivitätstabelle:')
print(df_sens.to_string(index=False))

## 13. Newsvendor-Bidding

Alle Modelle liefern Point Forecasts → direktes Gebot: **y_bid = ŷ**

Optimales Gebot: y* = F⁻¹(τ) mit τ = c_under / (c_under + c_over)

→ Für bessere Gebote: Quantile Regression oder probabilistische Erweiterung.

In [ ]:
C_OVER  = 10.0
C_UNDER = 15.0
tau     = C_UNDER / (C_UNDER + C_OVER)

print(f'Newsvendor tau = {tau:.3f} → optimales Quantil = {tau*100:.0f}%')
print()
print('── Newsvendor Loss Ranking ──────────────────────────────')
nv_df = pd.Series({
    name: newsvendor_loss(y_test, y_pred, C_OVER, C_UNDER)
    for name, y_pred in predictions.items()
}).sort_values()
print(nv_df.round(3).to_string())

## 14. Zusammenfassung

### Key Insights
1. **Windowing** wandelt das Zeitreihenproblem in Standard-Supervised-Learning um – funktioniert mit jedem ML-Modell
2. **W ist willkürlich** – Sensitivitätsanalyse zeigt wie stark W die Performance beeinflusst
3. **LR-Koeffizienten** zeigen direkt welche Lags am informativsten sind → interpretierbar
4. **Elastic Net** selektiert automatisch relevante Lags und setzt irrelevante auf 0
5. **RF / GBRT** profitieren von nichtlinearen Zusammenhängen zwischen Lags
6. **LSTM** verarbeitet die Sequenz nativ – kein Windowing im klassischen Sinne nötig
7. **Bestes RMSE ≠ bestes Newsvendor Loss** – zentrale Erkenntnis

### Offene TODOs
- [ ] Horizon H > 1 testen (Multi-Step Forecasting für Day-Ahead)
- [ ] Optimales W aus Sensitivitätsanalyse verwenden
- [ ] Zusatz-Features (Windgeschwindigkeit) Einfluss analysieren
- [ ] Quantile-Erweiterung für probabilistische Prognose